# 03 - Gobernanza con Unity Catalog

Este notebook documenta y demuestra la gobernanza del proyecto mediante Unity Catalog.

Objetivos:

- verificar el catálogo propio del proyecto y sus schemas Medallion;
- identificar el principal que ejecuta el notebook sin hardcodear usuarios;
- documentar un esquema de permisos de solo lectura sobre Silver;
- aplicar privilegios `USE CATALOG`, `USE SCHEMA` y `SELECT`;
- verificar los permisos mediante `SHOW GRANTS`;
- dejar evidencia reproducible de gobernanza para la rúbrica del proyecto.


## 1. Configuración

Se reutiliza el catálogo creado para el proyecto y los tres schemas de la arquitectura Medallion.

In [ ]:
from pathlib import Path
import sys

# Ejecutar desde el Git Folder completo, no desde un notebook importado aislado.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "accidentalidad" / "policies.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("No se encontró el paquete accidentalidad; abra el repositorio completo en un Git Folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from accidentalidad.policies import sql_identifier, parse_bool, nonnegative_int

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catálogo")
dbutils.widgets.text("reader_group", "", "Grupo de analistas existente")
dbutils.widgets.dropdown("apply_grants", "false", ["false", "true"], "Aplicar permisos al grupo")
dbutils.widgets.dropdown("verify_reader", "false", ["false", "true"], "Verificar usando identidad lectora")
PROY_CATALOG = sql_identifier(dbutils.widgets.get("catalog_name"))
PROY_SCHEMA_BRONZE, PROY_SCHEMA_SILVER, PROY_SCHEMA_GOLD = "bronze", "silver", "gold"
READER_GROUP = dbutils.widgets.get("reader_group").strip()
APPLY_GRANTS = parse_bool(dbutils.widgets.get("apply_grants"))
VERIFY_READER = parse_bool(dbutils.widgets.get("verify_reader"))
if APPLY_GRANTS and VERIFY_READER:
    raise ValueError("Ejecute la concesión y la verificación con identidades separadas")

## 2. Verificación de la estructura gobernada

El proyecto utiliza un catálogo propio y separa los datos por nivel de refinamiento. Esta separación facilita aplicar permisos distintos por capa.

In [ ]:
if not VERIFY_READER:
    print("Schemas disponibles en el catálogo:")
    display(spark.sql(f"SHOW SCHEMAS IN {PROY_CATALOG}"))

In [ ]:
if not VERIFY_READER:
    for schema in [PROY_SCHEMA_BRONZE, PROY_SCHEMA_SILVER, PROY_SCHEMA_GOLD]:
        print(f"\nTablas en {PROY_CATALOG}.{schema}:")
        display(spark.sql(f"SHOW TABLES IN {PROY_CATALOG}.{schema}"))

## 3. Identidad y grupos
Use un grupo de cuenta existente en `reader_group`. Un administrador ejecuta con
`apply_grants=true`. Después, un miembro lector sin privilegios heredados de escritura
abre el notebook con `apply_grants=false` y `verify_reader=true`.
El notebook no crea grupos ni revoca permisos previamente existentes.

In [ ]:
CURRENT_PRINCIPAL = spark.sql("SELECT current_user() AS principal").first()["principal"]

print(f"Principal actual: {CURRENT_PRINCIPAL}")


## 4. Política de acceso demostrada

Se aplica un patrón de **solo lectura sobre Silver**:

1. `USE CATALOG` permite resolver objetos dentro del catálogo.
2. `USE SCHEMA` permite acceder al namespace `silver`.
3. `SELECT` permite consultar las tablas del schema.

Esto ilustra el principio de mínimo privilegio: para consumir datos Silver no es necesario otorgar permisos de escritura ni administración.

In [ ]:
if APPLY_GRANTS:
    if not READER_GROUP:
        raise ValueError("Especifique un grupo existente en reader_group")
    principal_sql = READER_GROUP.replace("`", "``")
    sentencias = [
        f"GRANT USE CATALOG ON CATALOG {PROY_CATALOG} TO `{principal_sql}`",
        f"GRANT USE SCHEMA ON SCHEMA {PROY_CATALOG}.silver TO `{principal_sql}`",
        f"GRANT SELECT ON SCHEMA {PROY_CATALOG}.silver TO `{principal_sql}`",
    ]
    for sentencia in sentencias:
        spark.sql(sentencia)
    print(f"[OK] Permisos de lectura aplicados a {READER_GROUP}")
else:
    print("Modo inspección; no se modificaron permisos")

## 5. Evidencia de permisos

Se consultan los grants del catálogo y del schema Silver para dejar evidencia del control de acceso.

In [ ]:
if not VERIFY_READER:
    print(f"Grants sobre catálogo {PROY_CATALOG}:")
    display(spark.sql(f"SHOW GRANTS ON CATALOG {PROY_CATALOG}"))

In [ ]:
if not VERIFY_READER:
    print(f"Grants sobre schema {PROY_CATALOG}.{PROY_SCHEMA_SILVER}:")
    display(spark.sql(
        f"SHOW GRANTS ON SCHEMA {PROY_CATALOG}.{PROY_SCHEMA_SILVER}"
    ))

## 6. Validación de acceso de lectura

Se ejecuta una consulta sencilla sobre una tabla Silver para comprobar que el principal puede consumir la capa gobernada.

In [ ]:
if VERIFY_READER:
    denied = False
    try:
        # Operación sin filas: verifica MODIFY sin introducir registros.
        spark.sql(f"INSERT INTO {PROY_CATALOG}.silver.accidentes SELECT * FROM {PROY_CATALOG}.silver.accidentes WHERE 1 = 0")
    except Exception as error:
        # Solo aceptar denegación explícita, no errores de sintaxis/esquema/conectividad.
        error_class = error.getErrorClass() if hasattr(error, "getErrorClass") else ""
        if "INSUFFICIENT_PERMISSIONS" not in str(error_class) and "INSUFFICIENT_PRIVILEGES" not in str(error_class):
            raise
        denied = True
    if not denied:
        raise ValueError("La identidad puede escribir Silver; revise permisos directos, heredados y propiedad")
    display(spark.sql(f"SELECT COUNT(*) AS total_accidentes FROM {PROY_CATALOG}.silver.accidentes"))
    print("[OK] Lectura permitida y escritura denegada bajo la identidad actual")
else:
    print("La prueba de aislamiento queda pendiente de ejecutar con una identidad lectora")

## 7. Alcance de la evidencia
`SHOW GRANTS` permite inspeccionar concesiones. La prueba opcional comprueba lectura y
rechazo de una escritura sin filas bajo la identidad actual. La inspección con el propietario
no prueba mínimo privilegio. Complete y registre la prueba con una identidad lectora antes
de afirmar que la política está verificada.